# Yönetmelik Asistanı · Colab çalıştırıcı

Bu notebook yalnızca ince bir çalıştırıcıdır (plan 12.2); kodun tamamı depodadır.
Çalışma zamanı olarak **L4 GPU** seçin (Colab Pro; K-16: ana model Qwen3.5-27B). T4'te 4. hücre
yedek modeli (Qwen3.5-9B) kurar; ölçümler L4 ve 27B ile yapılır.

**Faz 0 hedefi:** ortamı kurmak ve `smoke` sağlık kontrolünü geçmek.

## 1 · GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 2 · Drive ve önbellek dizinleri

İndeks ve veriler her zaman Drive'da tutulur (oturum düşerse kaybolmasın diye).
Model dosyaları Drive'a yazılmaz: Qwen3.5-27B (~17 GB) tek başına ücretsiz Drive kotasını (15 GB)
aşar. Modeller her oturumda yeniden indirilir (27B ~5 dk).

**VS Code'da:** Drive'ı Colab sunucu menüsündeki *Mount Google Drive to server* ile bağlayın,
sonra bu hücreyi çalıştırın. Drive bağlı değilse hücre durur; böylece veriler yanlışlıkla
sunucunun geçici diskine yazılmaz.

In [ ]:
import os

DRIVE = '/content/drive/MyDrive'
if not os.path.isdir(DRIVE):
    try:  # Colab web arayüzünde çalışır; VS Code'da sunucu menüsünü kullanın
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f'drive.mount çalışmadı: {e}')
if not os.path.isdir(DRIVE):
    raise RuntimeError(
        "Google Drive bağlı değil. VS Code'da Colab sunucu menüsünden "
        "'Mount Google Drive to server' seçin ve bu hücreyi tekrar çalıştırın."
    )

BASE = '/content/drive/MyDrive/yonetmelik-asistani'
MODELLERI_DRIVEDA_TUT = False  # True: indirme süresi kısalır, Drive kotası dolar

os.environ['YA_DATA_DIR'] = f'{BASE}/data'
os.makedirs(os.environ['YA_DATA_DIR'], exist_ok=True)
if MODELLERI_DRIVEDA_TUT:
    os.environ['HF_HOME'] = f'{BASE}/hf-cache'
    os.environ['OLLAMA_MODELS'] = f'{BASE}/ollama-models'
print({k: os.environ.get(k) for k in ('YA_DATA_DIR', 'HF_HOME', 'OLLAMA_MODELS')})

## 3 · Depoyu al ve paketi kur

Depo herkese açık olduğu için token gerekmez. Depo özel yapılırsa Colab'ın
*Secrets* bölümüne `GITHUB_TOKEN` ekleyin.

In [ ]:
REPO_URL = 'https://github.com/Cnrygt11/yonetmelik-asistani.git'
REPO_DIR = '/content/yonetmelik-asistani'

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        REPO_URL = REPO_URL.replace('https://', f'https://{token}@')
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%pip install -q -e "{REPO_DIR}[ingest,search,dense,ui]"

## 4 · Ollama'yı kur, başlat ve modeli indir

Model GPU belleğine göre seçilir (K-16): 20 GB ve üstünde (L4) Qwen3.5-27B, altında (T4) Qwen3.5-9B.
Seçilen model sonraki bütün hücrelere `YA_LLM_MODEL` ile geçer. Çıktının sonundaki `ollama ps`
satırında PROCESSOR sütunu **100% GPU** olmalı; CPU yüzdesi varsa model belleğe sığmamıştır.

In [ ]:
import subprocess
GPU_MIB = int(subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
                             capture_output=True, text=True).stdout.split()[0])
MODEL = 'qwen3.5:27b-q4_K_M' if GPU_MIB >= 20000 else 'qwen3.5:9b'
ETIKET = '27b' if '27b' in MODEL else '9b'  # çıktı dosyası adlarında
os.environ['YA_LLM_MODEL'] = MODEL
print(f'GPU belleği {GPU_MIB} MiB → {MODEL}')
!bash {REPO_DIR}/scripts/colab_setup.sh {MODEL}
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 5 · Sağlık kontrolü

Beş soru çalışır: dördü Türkçe akıcılık ve hız, biri boş bağlamda ret davranışı.
Rapor oturum kapanınca kaybolmasın diye Drive'a yazılır. Faz 0 çıkışında bu raporu
depodaki `eval/reports/` klasörüne ekleyin.

In [ ]:
from datetime import datetime
os.makedirs(f'{BASE}/reports', exist_ok=True)
REPORT = f"{BASE}/reports/smoke_{datetime.now():%Y%m%d_%H%M%S}.json"
!cd {REPO_DIR} && python -m yonetmelik_asistani.smoke --model {MODEL} --report {REPORT}
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 6 · Faz 2: belgeler, indeks ve arama değerlendirmesi

Bu bölüm **Ollama gerektirmez**; 4 ve 5 atlanabilir (1-3 çalıştırılmış olmalı).
Belgeler Drive'da (`YA_DATA_DIR`), indeks Colab'ın yerel diskinde tutulur: Qdrant'ın SQLite
dosyası Drive'da kilitlenebiliyor. İndeksin yeniden kurulması yaklaşık bir dakika sürer.

**6.1** Drive'daki `data/raw` belgelerinden metni çıkar ve doğrula. Colab internete çıkmaz:
`.gov.tr` siteleri Colab'dan erişilemiyor. İndirme ve çapraz kontrol yerel bilgisayarda yapılır,
güncel `data/raw` klasörü Drive'a yüklenir. `extract` her dosyanın sha256 özetini
`configs/corpus_lock.json` ile karşılaştırır.

In [ ]:
os.environ['YA_INDEX_DIR'] = '/content/ya-index'
!cd {REPO_DIR} && python -m yonetmelik_asistani.ingest extract

**6.2** Parçala, BM25 ve BGE-M3 indekslerini kur (BGE-M3 ilk seferde ~2,3 GB indirir).

In [ ]:
!cd {REPO_DIR} && python -m yonetmelik_asistani.chunking && python -m yonetmelik_asistani.index

**6.3** Geliştirme setinde dört arama kipini karşılaştır (reranker ilk seferde ~2,3 GB indirir).
Çıktının tamamını paylaşın; rapor Drive'a da kopyalanır.

In [ ]:
!cd {REPO_DIR} && python eval/run_eval.py --methods bm25,dense,hybrid,hybrid_rerank --tokenizers char --sozluk ikisi
!mkdir -p {BASE}/reports && cp {REPO_DIR}/eval/reports/retrieval_*.json {BASE}/reports/
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

**6.4** Örnek arama (her aşamanın sırası ve skoru görünür).

In [ ]:
!cd {REPO_DIR} && python -m yonetmelik_asistani.retrieval "Kaydımı dondurmak istiyorum, ne zamana kadar başvurmalıyım?" -k 3

## 7 · Uçtan uca deneme (Faz 4 öncesi)

Aynı GPU'da Ollama (4'te seçilen model) + BGE-M3 + reranker birlikte çalışır ve 6 soruya kaynak gösteren
cevap üretilir. Amaç ölçmek değil; bellek, süre, kaynak etiketleri ve ret davranışını erken görmek.

**Önce:** 1-3, ardından **4** (Ollama kurulumu ve model indirme, ~5 dk) çalıştırılmış olmalı.
İndeks yoksa 6.1 ve 6.2 de gerekir. Rapor Drive'a yazılır (`reports/e2e_*.json`).

In [ ]:
from datetime import datetime
os.environ['YA_INDEX_DIR'] = '/content/ya-index'
if not os.path.isdir('/content/ya-index/qdrant'):
    print('İndeks yok: önce 6.1 ve 6.2 çalıştırılmalı')
else:
    E2E = f"{BASE}/reports/e2e_{datetime.now():%Y%m%d_%H%M%S}.json"
    !cd {REPO_DIR} && python eval/e2e_deneme.py --report {E2E}

## 8 · Faz 4: geliştirme setinde tam ölçüm

Geliştirme bölümündeki 134 soruya cevap üretilir (varsayılan istem v3, 4'te seçilen model, soruların
fakülte profiliyle). Süre: L4 ve 27B ile soru başına ~25 sn, toplam ~1 saat.
**Önce:** 1-3, **4** (Ollama), 6.1-6.2 (indeks).

Her cevap üretildiği anda Drive'daki dosyaya yazılır. Oturum düşerse aynı hücreyi yeniden çalıştırın:
kaldığı yerden devam eder (ayarlar değişmişse betik durur ve yeni dosya ister). Dosya adı tarih ve
modeli içerir; bitince `reports/` klasöründeki dosyayı indirip `eval/reports/` klasörüne koyun.

In [ ]:
from datetime import date
os.environ['YA_INDEX_DIR'] = '/content/ya-index'
CEVAPLAR = f"{BASE}/reports/cevaplar_gelistirme_v3_{ETIKET}_{date.today():%Y%m%d}.jsonl"
print(CEVAPLAR)
!cd {REPO_DIR} && python eval/run_answers.py --out {CEVAPLAR}

## 9 · Arayüz ve canlı test

Gradio arayüzünü açar: fakülte ve düzey seçimi, soru, cevap, açılır kaynak paneli ve cevabın altında
değerlendirme (Doğru / Kısmen / Yanlış / Emin değilim, isteğe bağlı not). Her soru, cevap ve
değerlendirme Drive'daki `reports/arayuz_kayit.jsonl` dosyasına eklenir (her oturum aynı dosyaya).
**Önce:** 1-3, 4 (Ollama), indeks (6.1-6.2). Modeller yüklenince (~1 dk) çıktıda
`Running on public URL: https://….gradio.live` satırı görünür; o bağlantıyı açın. Hücre çalıştığı
sürece arayüz açık kalır; bitirince hücreyi durdurun. Bağlantıyı bilen herkes açabilir ve soruları
kaydedilir: yalnızca güvendiğiniz kişilerle paylaşın.

In [ ]:
os.environ['YA_INDEX_DIR'] = '/content/ya-index'
KAYIT = f"{BASE}/reports/arayuz_kayit.jsonl"
!cd {REPO_DIR} && python -m yonetmelik_asistani.ui --share --kayit {KAYIT}

## 10 · Faz 4 iyileştirme: aday dökümü ve sabit 40 soru

**10.1** Aday dökümü: her geliştirme sorusu için reranker'ın skorladığı 30 aday skorlarıyla yazılır
(arama değiştiğinde bağlam ayarlarını yerelde yeniden taramak için). İki döküm çıkar: Türkçe
karakter geri getirme açık (`_tr_acik`) ve kapalı (`_tr_kapali`). **Ollama gerekmez**
(1-3 ve 6.1-6.2 yeterli), ~5 dk. Bitince `reports/` klasöründeki iki `adaylar_gelistirme_tr_*.jsonl`
dosyasını indirip `eval/reports/` klasörüne koyun.

In [ ]:
os.environ['YA_INDEX_DIR'] = '/content/ya-index'
!cd {REPO_DIR} && python eval/dump_candidates.py --turkce-karakter ikisi --out {BASE}/reports/adaylar_gelistirme.jsonl

**10.2** Sabit 40 soru (`eval/alt_set_40.txt`), istem v3, 4'te seçilen model. L4 ve 27B ile ~20 dk.
Hızlı karşılaştırma içindir; asıl ölçüm 8'dir. Yarıda kalırsa aynı hücre kaldığı yerden devam eder.

In [ ]:
from datetime import date
os.environ['YA_INDEX_DIR'] = '/content/ya-index'
ALT40 = f"{BASE}/reports/cevaplar_alt40_v3_{ETIKET}_{date.today():%Y%m%d}.jsonl"
print(ALT40)
!cd {REPO_DIR} && python eval/run_answers.py --ids-file eval/alt_set_40.txt --out {ALT40}